# Tutorial 7b — Two-model coupling II: sampling the joint

Estimated time: 20-25 minutes. Steps 1-2 need no backend; Steps 3-6 need PyMC.

> **Module 7, part 2 of 3.** 7a said what a coupling *is*. This part is about actually
> drawing from the coupled distribution — and about the harder question of whether the
> sampler that produced your numbers explored anything at all.
>
> | | Question | Needs |
> |---|---|---|
> | 7a | What does it mean to couple two models? | nothing extra |
> | **7b (you are here)** | How is the coupled distribution sampled, and can I trust it? | PyMC, from Step 3 |
> | 7c | What can I ask it, once coupled? | PyMC |

## Where 7a left off

7a established three things this part builds on directly:

1. A coupling is a **factor** multiplied into a product of priors — `p(y)p(C)·φ(y,C)` —
   which reshapes a round cloud into a diagonal ridge.
2. The default `--method propagate` does *not* sample that product. It draws from the priors
   and overwrites the coupling's target, so information flows one way and the surrogates are
   never consulted.
3. The stored artifact says which happened: `method` and `surrogates_evaluated`.

The obvious next question is what the *other* method does, and whether you should believe it.

## What you'll be able to do afterwards

- **Sample the coupled joint properly**, and see both ends of a coupling move.
- **Check a sampler against an answer known on paper**, rather than against itself.
- **Read ESS, r-hat and divergences**, and say when a result is not yet reportable.
- **Recognise the trap**: a chain that never moved, reporting a healthy acceptance rate.

## How you'll know you got it

Given a coupled model and a run, you can say whether the numbers are trustworthy — and if
not, which diagnostic told you and what to change.


In [ ]:
# Cross-platform setup — the same opening cell as every other tutorial.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "src" / "bayesian_metamodeling").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from bayesian_metamodeling.tutorial import bootstrap, run_mm_cli  # noqa: E402

root = bootstrap()
ROOT = root
print("Repo root:", root)

# Steps 1-2 need no optional backend. Steps 3-6 compare samplers and fit a surrogate.
try:
    import pymc  # noqa: F401
    PYMC_AVAILABLE = True
    print("PyMC found — every step below will run.")
except ImportError:
    PYMC_AVAILABLE = False
    print("\nNote: PyMC is absent, so Steps 3-6 will skip with a printed reason.")
    print("Steps 1-2 run without it. To get the rest:")
    print("  conda env create -f environment-all.yml && conda activate py312_bayesmm_all")


## Step 1: the other half of `meta sample` — and why it refuses here

Everything above used the default, `--method propagate`. It is worth stating exactly what
that does, because it is not what "sample the coupled model" usually means:

1. draw every variable **independently from its prior**, then
2. overwrite each coupled target with `transform(source)` (+ noise for a soft link).

So a coupling reshapes its **target** and leaves its **source** exactly at the prior.
Information flows one way. That is fast and genuinely useful for "what does this coupling
imply downstream?", and it is *not* inference.

`--method joint` targets the actual joint density — priors, couplings **and** the surrogate
likelihoods together — with a Metropolis chain. There a coupling is evidence about the
**pair**, so it tightens both variables.

| | `propagate` (default) | `joint` |
|---|---|---|
| What it does | prior draws + post-draw transform | Metropolis on the joint log-density |
| Surrogate likelihoods | not evaluated (`surrogates={}`) | conditioned on |
| Effect on the coupling's *source* | none — stays at its prior | tightened |
| Deterministic targets | assigned after the draw | recomputed from source each step |
| Recorded as | `method: "prior_propagation"` | `method: "random_walk_metropolis"` |
| Speed | instant | seconds to minutes |

**Why Metropolis and not NUTS:** a fitted surrogate's `log_prob` is a black box with no
gradient, so a gradient-free sampler is what the model admits. That costs efficiency, not
correctness. Expressing the whole model as a PyTensor graph would unlock NUTS, and it needs
every surrogate backend to provide a symbolic `log_prob` — which none does today.

### First: ask for it here, and watch it decline

This spec's surrogate artifacts are the placeholders described at the top of this notebook —
signatures with no fitted model. `joint` conditions on the surrogate likelihoods, so it
cannot proceed, and it says so instead of silently degrading to priors-plus-couplings. Run
it and read the message: a good error names the cause, the consequence and the two ways out.

In [ ]:
# Ask for the real thing on THIS spec. A non-zero exit is the expected, informative
# outcome here, so `check=False` — we want the message, not an exception.
exit_code = run_mm_cli(
    "meta", "sample", "tutorials/specs/metamodel.two_model.pymc.json",
    "--draws", "50", "--tune", "20", "--chains", "1", "--seed", "7",
    "--method", "joint",
    check=False,
)
print(f"\nexit code = {exit_code}   (1 = refused, having told you exactly why)")
assert exit_code == 1, (
    "Expected `--method joint` to refuse on placeholder surrogate artifacts. If this "
    "passed, either the artifacts gained a backend_payload or the guard in "
    "meta/joint_sampling.py::load_surrogates_for_ir stopped firing."
)

That refusal is a design decision worth copying. A missing surrogate would reduce the joint
density to priors-plus-couplings, and the chain would run happily and return numbers that
*look* like a posterior. Loudly refusing beats quietly answering a different question.

### Then: see joint sampling work, with a right answer to check against

Since this spec cannot run it, the cell below builds a **two-variable model small enough to
have a closed-form answer**: normal priors and a linear-Gaussian coupling make the joint
Gaussian, so we can compute exactly what the sampler *should* produce and print it next to
what it does.

That is the check worth internalising — not "did it return numbers" but "did it return the
**right** numbers". Any time your model is small enough to have an analytic answer, sample
it anyway and compare. It is the cheapest correctness test in Bayesian workflow, and the
only one that catches a sampler that is confidently wrong.

(`projects/tcr_signaling/notebooks/03_metamodel_inference.ipynb` runs `--method joint` for
real, conditioned on four fitted surrogates.)

## Step 2: the coupled joint, checked against pen and paper

The spec above cannot be sampled jointly, because its surrogates are placeholders. But the
*coupling machinery* can be — and on a two-variable model the right answer is available in
closed form, so the sampler can be checked against the truth rather than against itself.

This runs with no optional backend: the random-walk sampler is pure numpy.


In [ ]:
import numpy as np

from bayesian_metamodeling.meta.compiler import compile_metamodel
from bayesian_metamodeling.meta.ir import (
    CouplingFactorIR,
    MetamodelIR,
    PriorFactorIR,
    VariableIR,
)
from bayesian_metamodeling.meta.joint_sampling import sample_joint

# x ~ N(1.0, 0.5);  y ~ N(0.0, 2.0);  and a coupling asserting y ~ N(1.5x - 0.4, 0.3)
mx, sx, my, sy, alpha, beta, sigma_j = 1.0, 0.5, 0.0, 2.0, 1.5, -0.4, 0.3

ir = MetamodelIR(
    name="two_variable_demo",
    variables=[VariableIR(name="x"), VariableIR(name="y")],
    factors=[
        PriorFactorIR(variable="x", distribution={"kind": "normal", "loc": mx, "scale": sx}),
        PriorFactorIR(variable="y", distribution={"kind": "normal", "loc": my, "scale": sy}),
        CouplingFactorIR(
            coupling_type="gaussian_link", source="x", target="y",
            transform={"kind": "affine", "alpha": alpha, "beta": beta}, sigma=sigma_j,
        ),
    ],
)

# Closed form: collecting the quadratic form in (x, y) gives precision Lambda and
# linear term h, so mean = Lambda^-1 h and cov = Lambda^-1.
lam = np.array([[1/sx**2 + alpha**2/sigma_j**2, -alpha/sigma_j**2],
                [-alpha/sigma_j**2, 1/sy**2 + 1/sigma_j**2]])
h = np.array([mx/sx**2 - alpha*beta/sigma_j**2, my/sy**2 + beta/sigma_j**2])
cov = np.linalg.inv(lam)
exact_mean, exact_sd = cov @ h, np.sqrt(np.diag(cov))
exact_corr = cov[0, 1] / (exact_sd[0] * exact_sd[1])

samples, diag = sample_joint(compile_metamodel(ir), draws=8000, tune=2000, chains=4, seed=11)
xs, ys = samples["x"].ravel(), samples["y"].ravel()
corr = float(np.corrcoef(xs, ys)[0, 1])

print(f"{'':10} {'prior':>10} {'joint (sampled)':>18} {'joint (exact)':>15}")
print(f"{'mean x':10} {mx:10.3f} {xs.mean():18.3f} {exact_mean[0]:15.3f}")
print(f"{'mean y':10} {my:10.3f} {ys.mean():18.3f} {exact_mean[1]:15.3f}")
print(f"{'sd x':10} {sx:10.3f} {xs.std(ddof=1):18.3f} {exact_sd[0]:15.3f}")
print(f"{'sd y':10} {sy:10.3f} {ys.std(ddof=1):18.3f} {exact_sd[1]:15.3f}")
print(f"{'corr':10} {0.0:10.3f} {corr:18.3f} {exact_corr:15.3f}")

# --- grade it -------------------------------------------------------------------------
# Tolerances are set from measured Monte Carlo scatter, not from one lucky run: over 15
# seeds at these settings the worst deviations from the exact values were 0.064 (means),
# 0.038 (sds) and 0.007 (corr). The tolerances below leave ~2.5x headroom on that.
# They still have teeth, because a sampler that ignored the coupling and returned the
# prior would miss by 0.95 (mean y), 1.25 (sd y) and 0.92 (corr) — 6x to 23x these bounds.
TOL_MEAN, TOL_SD, TOL_CORR = 0.15, 0.10, 0.04
assert abs(xs.mean() - exact_mean[0]) < TOL_MEAN, f"mean x off by {abs(xs.mean()-exact_mean[0]):.3f}"
assert abs(ys.mean() - exact_mean[1]) < TOL_MEAN, f"mean y off by {abs(ys.mean()-exact_mean[1]):.3f}"
assert abs(xs.std(ddof=1) - exact_sd[0]) < TOL_SD, f"sd x off by {abs(xs.std(ddof=1)-exact_sd[0]):.3f}"
assert abs(ys.std(ddof=1) - exact_sd[1]) < TOL_SD, f"sd y off by {abs(ys.std(ddof=1)-exact_sd[1]):.3f}"
assert abs(corr - exact_corr) < TOL_CORR, f"corr off by {abs(corr-exact_corr):.3f}"

devs = {
    "mean x": abs(xs.mean() - exact_mean[0]),
    "mean y": abs(ys.mean() - exact_mean[1]),
    "sd x": abs(xs.std(ddof=1) - exact_sd[0]),
    "sd y": abs(ys.std(ddof=1) - exact_sd[1]),
    "corr": abs(corr - exact_corr),
}
worst = max(devs, key=lambda k: devs[k])
print(
    f"\nGraded against the closed form: all five inside {TOL_MEAN}/{TOL_SD}/{TOL_CORR};"
    f" the largest gap is {worst}, off by {devs[worst]:.3f}."
)
print(
    "That gap is Monte Carlo error, not a bug. These are random-walk Metropolis draws with"
)
print(
    f"corr(x, y) ≈ {corr:.2f}, so consecutive draws are strongly autocorrelated and the raw"
)
print("draw count is not the sample size:")
ess = diag.get("ess") or {}
if {"x", "y"} <= set(ess):
    print(
        f"    effective sample size ≈ {ess['x']:.0f} for x and {ess['y']:.0f} for y,"
        f" out of {xs.size} raw draws"
    )
    print(
        f"    — a ~{xs.size / ess['y']:.0f}x reduction, and that is what sets the error bar on"
    )
    print("    every number in the table above. A healthy accept_rate does not imply the")
    print("    chain went anywhere; effective sample size is what says how far it got.")
print("Gaps of this size are expected, and they shrink like 1/sqrt(ESS) — so buying a decimal")
print("place costs 100x the draws. A 20% gap would mean something is actually wrong.")

print(f"\naccept_rate = {diag['accept_rate']:.2f}   (the sampler's target_accept is 0.3)")
print(
    f"Tuning found proposal widths that get about {100 * diag['accept_rate']:.0f}% of moves"
    " accepted."
)
print("That band is deliberate: the classical optimum for random-walk Metropolis is ≈0.44 in")
print("one dimension and ≈0.23 in high dimensions, and this sampler moves one coordinate at a")
print("time, so 0.3 sits sensibly in between. Near 0.01 the proposals are too big and almost")
print("everything is rejected; near 0.95 they are too small and the chain barely moves. Both")
print("still return draws — neither returns the distribution.")

print(
    f"\nRead the y row: prior sd {sy:.2f} -> joint sd {ys.std(ddof=1):.3f} (exact "
    f"{exact_sd[1]:.3f}). The coupling carried information from x into y."
)
print(
    f"And read the x row: the SOURCE moved too, prior mean {mx:.2f} -> joint mean "
    f"{xs.mean():.3f} (exact {exact_mean[0]:.3f}), prior sd {sx:.2f} -> joint sd "
    f"{xs.std(ddof=1):.3f}."
)
print("Under `propagate`, x would have stayed at exactly N(1.00, 0.50). That is the difference.")

**Now apply the Step 2 field test to this output.** `x` is the coupling's source. Its prior
sd is 0.50 and the sampled sd printed above is smaller — the source moved. That single
comparison, source width against prior width, is how you tell inference from propagation
without reading a line of source code.

It is also why `inference_data.json` records `method` and `surrogates_evaluated`. Six months
from now, a `samples_dataset.json` on disk looks the same either way; the sidecar is the
only thing that says which question it answers.

## Step 3: a faster sampler — and why you check it before using it

Step 2 sampled the coupled model with a **random walk** — propose a small step, accept or
reject, repeat. It works for any surrogate, because all it ever needs is to *evaluate* the
model at a point.

This notebook uses **NUTS** instead (No-U-Turn Sampler), the standard modern algorithm,
which follows the shape of the distribution using its gradient. Faster, but only usable when
the model can be differentiated — 7c returns to when that holds.

Two samplers, so the first question is whether they compute the same thing. The model here
is T7's two-variable example, chosen because its answer can be worked out **on paper**: with
normal priors and a linear coupling, the joint is a 2-D Gaussian whose mean and spread come
from inverting a 2×2 matrix. The closed form is the referee.


In [ ]:
import numpy as np

if not PYMC_AVAILABLE:
    print("Step 1 SKIPPED — needs PyMC (see preflight above).")
    nuts_s = rw_s = None
else:
    from bayesian_metamodeling.meta.compiler import compile_metamodel
    from bayesian_metamodeling.meta.ir import (
        CouplingFactorIR, MetamodelIR, PriorFactorIR, VariableIR,
    )
    from bayesian_metamodeling.meta.joint_sampling import sample_joint
    from bayesian_metamodeling.meta.nuts_sampling import sample_nuts

    mx, sx, my, sy, alpha, beta, sigma = 1.0, 0.5, 0.0, 2.0, 1.5, -0.4, 0.3
    pair = MetamodelIR(
        name="pair",
        variables=[VariableIR(name="x"), VariableIR(name="y")],
        factors=[
            PriorFactorIR(variable="x", distribution={"kind": "normal", "loc": mx, "scale": sx}),
            PriorFactorIR(variable="y", distribution={"kind": "normal", "loc": my, "scale": sy}),
            CouplingFactorIR(coupling_type="gaussian_link", source="x", target="y",
                             transform={"kind": "affine", "alpha": alpha, "beta": beta},
                             sigma=sigma),
        ],
    )

    lam = np.array([[1/sx**2 + alpha**2/sigma**2, -alpha/sigma**2],
                    [-alpha/sigma**2, 1/sy**2 + 1/sigma**2]])
    h = np.array([mx/sx**2 - alpha*beta/sigma**2, my/sy**2 + beta/sigma**2])
    cov = np.linalg.inv(lam)
    exact_mean, exact_sd = cov @ h, np.sqrt(np.diag(cov))

    nuts_s, nuts_d = sample_nuts(compile_metamodel(pair), draws=3000, tune=1000,
                                 chains=2, seed=11)
    rw_s, rw_d = sample_joint(compile_metamodel(pair), draws=9000, tune=2500,
                              chains=2, seed=11)

    print(f"{'':<24}{'mean x':>9}{'mean y':>9}{'sd x':>9}{'sd y':>9}")
    print(f"{'on paper (the truth)':<24}{exact_mean[0]:>9.3f}{exact_mean[1]:>9.3f}"
          f"{exact_sd[0]:>9.3f}{exact_sd[1]:>9.3f}")
    for label, s in (("NUTS (this notebook)", nuts_s), ("random walk (T7)", rw_s)):
        print(f"{label:<24}{s['x'].mean():>9.3f}{s['y'].mean():>9.3f}"
              f"{s['x'].std():>9.3f}{s['y'].std():>9.3f}")


**All three rows agree.** That is the licence to use the faster one: it is not an
approximation or a different model, it is the same distribution explored more cleverly.

Keep the habit whenever a pipeline offers you a shortcut — a problem with a known answer is
the cheapest possible check, and it costs one cell.


## Step 4: what the gradients buy

Both samplers found the right answer, so why bother?

Because *number of draws* and *amount of information* are not the same thing. A random walk
proposes a step in a random direction. If the distribution is shaped like a **narrow
diagonal ridge** — and coupled models produce exactly that — then almost
every random direction points off the ridge and gets rejected. The draws you keep are
strongly correlated, so a thousand of them may be worth only a handful of independent ones.
**Effective sample size (ESS)** is that count.


In [ ]:
if not PYMC_AVAILABLE or nuts_s is None:
    print("Step 2 SKIPPED — needs Step 1.")
else:
    n_nuts, n_rw = nuts_s["x"].size, rw_s["x"].size
    print(f"{'':<20}{'draws':>8}{'ESS(x)':>9}{'per draw':>10}")
    print(f"{'NUTS':<20}{n_nuts:>8}{nuts_d['ess']['x']:>9.0f}{nuts_d['ess']['x']/n_nuts:>9.0%}")
    print(f"{'random walk':<20}{n_rw:>8}{rw_d['ess']['x']:>9.0f}{rw_d['ess']['x']/n_rw:>9.0%}")
    print()
    print("Diagnostics NUTS reports that a random walk cannot:")
    print(f"  r-hat (max over variables) = {max(nuts_d['r_hat'].values()):.4f}"
          "   — do independent chains agree? >1.01 means no")
    print(f"  divergences                = {nuts_d['divergences']}"
          "        — geometry the sampler could not follow; >0 means distrust this")


On this two-variable toy the difference hardly matters. On the real four-surrogate
metamodel in `projects/tcr_signaling` the same comparison is **0.7% against 58%**, and three
variables came back from the random walk with an effective sample size in the *tens* —
summaries that looked perfectly reasonable and meant nothing.

**This is not a detail of our implementation.** The paper this repository reproduces used
the No-U-Turn sampler for exactly this step, and its central result turns out to be a ridge
— which is precisely the shape a coordinate-wise random walk cannot climb. 7c derives it.

Those last two names are worth pinning down, since they are how you decide whether to
believe a run at all:

- **r-hat** runs the sampler as several independent chains from different starting points
  and asks whether they ended up describing the same distribution. If they agree it sits at
  1.00; above about 1.01 they disagree, which means at least one of them has not explored
  the whole distribution and the summaries are premature.
- **Divergences** are steps the sampler could not compute reliably, usually because the
  distribution has a region far sharper than the step size it settled on. Even a handful
  means the result is biased in a way more draws will not fix.

Neither is a score to maximise; both are alarms. A random walk can raise neither.

> **r-hat needs at least two chains.** With one chain it is undefined, and the framework
> reports `None` rather than a number — a NaN would quietly satisfy any `r_hat < 1.01` check
> and make a single-chain run look flawless.


## Step 5 — the arrow this framework exists for: a *fitted* surrogate doing inference

Everything so far coupled variables to each other. Step 1's refusal was honest but
one-sided: `--method joint` declined placeholder artifacts, and you have not yet seen it
succeed on a real one. That is the arrow the whole pipeline is built around —
**sweep -> surrogate -> metamodel -> inference** — and this is where it closes.

The setup is small enough to check by hand:

- a simulator with **real noise**: `y = a + b + N(0, 0.3)`;
- a `pymc_gp` surrogate fitted to 80 runs of it;
- priors that **disagree** with it on purpose — `a ~ N(1, 0.6)` and `b ~ N(1, 0.6)` put
  `a + b` near 2, while `y ~ N(4, 0.4)` insists `y` is near 4.

Something has to give. Predict before you run: does `y` come down, do `a` and `b` go up,
or both? Under `propagate` the question could not even arise — the surrogate is never
consulted, `y` is simply overwritten, and `a` and `b` stay at their priors.


In [ ]:
if not PYMC_AVAILABLE:
    print("Step 4 SKIPPED — fitting a surrogate needs PyMC (see the note above).")
    samples_f = diag_f = None
else:
    import numpy as np

    from bayesian_metamodeling.meta.compiler import compile_metamodel
    from bayesian_metamodeling.meta.ir import (
        MetamodelIR, PriorFactorIR, SurrogateLikelihoodFactorIR, VariableIR,
    )
    from bayesian_metamodeling.meta.joint_sampling import sample_joint
    from bayesian_metamodeling.surrogates.backends import fit_backend_model

    # 1. A noisy simulator, and a surrogate fitted to 80 of its runs.
    _rng = np.random.default_rng(0)
    _a = _rng.uniform(0.0, 2.0, 80)
    _b = _rng.uniform(0.0, 2.0, 80)
    _y = (_a + _b + _rng.normal(0.0, 0.3, 80)).reshape(-1, 1)

    surrogate = fit_backend_model(
        backend="pymc_gp", x=np.column_stack([_a, _b]), y=_y,
        input_names=["a", "b"], output_names=["y"],
        backend_config={"draws": 400, "tune": 400, "chains": 1, "target_accept": 0.9},
        seed=0,
    )
    _probe = np.asarray(
        surrogate.sample(inputs={"a": np.array([1.0]), "b": np.array([1.0])}, n=500, seed=1)
    ).reshape(-1)
    print(f"surrogate at (a=1, b=1):  mean={_probe.mean():.3f}  sd={_probe.std():.3f}")
    print("  (that sd is the simulator's noise, learned. Hold on to it - Step 5 needs it.)")

    # 2. A metamodel whose priors disagree with the surrogate.
    ir_fitted = MetamodelIR(
        name="fitted_two_model",
        variables=[VariableIR(name=n) for n in ("a", "b", "y")],
        factors=[
            PriorFactorIR(variable="a", distribution={"kind": "normal", "loc": 1.0, "scale": 0.6}),
            PriorFactorIR(variable="b", distribution={"kind": "normal", "loc": 1.0, "scale": 0.6}),
            PriorFactorIR(variable="y", distribution={"kind": "normal", "loc": 4.0, "scale": 0.4}),
            SurrogateLikelihoodFactorIR(surrogate_ref="toy", inputs=["a", "b"], outputs=["y"]),
        ],
    )

    samples_f, diag_f = sample_joint(
        compile_metamodel(ir_fitted), draws=6000, tune=2000, chains=2, seed=5,
        surrogates={"toy": surrogate},
    )

    print(f"\naccept_rate={diag_f['accept_rate']:.2f}   worst ESS={min(diag_f['ess'].values()):.0f}"
          f"   poorly mixed: {diag_f['poorly_mixed'] or 'none'}")
    print(f"\n{'var':<4}{'prior':>14}{'joint mean':>12}{'joint sd':>10}")
    for _name, (_mu, _sd) in (("a", (1.0, 0.6)), ("b", (1.0, 0.6)), ("y", (4.0, 0.4))):
        _d = np.asarray(samples_f[_name], dtype=float).ravel()
        print(f"{_name:<4}{f'N({_mu}, {_sd})':>14}{_d.mean():>12.3f}{_d.std():>10.3f}")

    _am = float(np.asarray(samples_f["a"], float).mean())
    _bm = float(np.asarray(samples_f["b"], float).mean())
    _ym = float(np.asarray(samples_f["y"], float).mean())
    print(f"\na + b = {_am + _bm:.3f}   vs   y = {_ym:.3f}   -> they agree to "
          f"{abs(_am + _bm - _ym):.3f}")


**Read the table.** `a` and `b` started with prior means of 1.0 and came out higher; `y`
started at 4.0 and came down. Neither side won — `a + b` and `y` met, and that meeting *is*
the inference. This is the asymmetry from earlier disappearing: under `propagate`, `y` would
have been overwritten and `a`, `b` would have sat at exactly their priors, learning nothing.

Note also that `a` and `b` **narrowed** relative to their priors. The surrogate is evidence
about them, not only about `y` — information flowed backwards through the likelihood, which
is the thing a coupling alone cannot do under the default method.

And check `worst ESS` before believing any of it. The next step explains why that number is
printed at all.


## Step 6 — why your surrogate's noise is load-bearing

Refit the surrogate above on **noiseless** data (`y = a + b` exactly, no `N(0, 0.3)`) and
this stops working — quietly. `pymc_gp` fits the plane to machine precision, its predictive
sd falls to ~1e-5, and its likelihood becomes very nearly a delta function.

The posterior then lives on a razor-thin ridge: `y` must equal `a + b` to five decimals, so
any move in `a` that is not matched *in the same step* by `y` is rejected. A random walk
proposes one coordinate at a time, so it essentially never proposes such a move. Tuning
responds by shrinking every proposal until something is accepted, and the chain inches along
at ~1e-5 per step through a distribution that is genuinely ~0.6 wide.

What that looks like, if nothing warned you:

| var | prior sd | joint sd |
|---|---|---|
| a | 0.60 | **0.000** |
| b | 0.60 | **0.000** |
| y | 3.00 | **0.000** |

Those zeros read like conditioning that pinned the variables precisely. They mean the
sampler went nowhere. And `accept_rate` was **0.29** — a perfectly healthy-looking number,
because every one of those microscopic moves was accepted.

That is why `sample_joint` reports per-variable effective sample size, and why
`bayesmm meta sample --method joint` prints a warning naming any variable that barely moved.
Three things to take away:

1. **Read ESS, not `accept_rate`.** A high acceptance rate is entirely compatible with a
   chain that never left its starting point.
2. **Do not fit a surrogate to noiseless simulator output and expect downstream inference to
   behave.** If your simulator is deterministic, the width your surrogate reports is
   *misfit*, not noise — T5's product channel is exactly that case — and driving it toward
   zero is not a sign of a good fit.
3. **A gradient-based sampler would not have this problem**, which is the strongest argument
   for expressing surrogate `log_prob` as a PyTensor graph. It is the same prerequisite as
   unlocking NUTS.

The real TCR metamodel in `projects/tcr_signaling` trips this warning today, on three of its
fourteen variables. Knowing to look is the difference between reporting a result and
reporting an artefact.


## Recap: what 7b established

- **`--method joint` samples the product** 7a wrote down — priors, couplings and surrogate
  likelihoods together — so both ends of a coupling move and the surrogates are conditioned
  on. It refuses placeholder artifacts out loud rather than quietly dropping their factors.
- **Check a sampler against a known answer.** The two-variable coupled joint is Gaussian and
  its mean and spread come from inverting a 2×2 matrix; both samplers reproduce it.
- **`--method nuts` samples the same distribution with gradients.** Same answer, far more
  information per draw, plus r-hat and divergence counts. It applies when every surrogate is
  `pymc_gp`; 7c explains why that condition exists.
- **Acceptance rate is not evidence of anything.** A chain can accept 30% of its proposals
  and still never leave its starting point. Read ESS; read r-hat when you have ≥2 chains.
- **A surrogate fitted to noiseless output breaks downstream inference**, because its
  likelihood becomes a near-delta and the posterior collapses onto a ridge a random walk
  cannot follow. If your simulator is deterministic, the width your surrogate reports is
  *misfit*, not noise.

**Next — 7c** uses all of this to ask the question the whole framework exists for: given
something you measured, what does it imply about a parameter in a different model?


## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `Joint sampling unavailable: ... has no 'backend_payload'` | Expected in Step 1 — this spec's surrogates are placeholders, not fits | Use `--method propagate`, or point the spec at a real artifact from `bayesmm surrogate fit` (T5/T6) |
| `Step 5 SKIPPED — fitting a surrogate needs PyMC` | no PyMC in this kernel | `conda env create -f environment-all.yml`, then use the `py312_bayesmm_all` kernel |
| `accept_rate` looks fine but the posterior is suspiciously narrow | the chain may not have moved | check ESS per variable; see Step 6 |
| `r_hat` is `None` | one chain, where r-hat is undefined | use `chains=2` or more; the framework refuses to report a NaN that would pass every check |
| `divergences > 0` | geometry the sampler could not integrate | raise `target_accept`, or look for a coupling `sigma` far tighter than the priors it constrains |
| the two samplers disagree | they should not — they encode the same density | check you did not change the spec between runs; then report it, because that is a real bug |


## Final check

Asserts what 7b claims, each in a way that could fail on a run that executed but demonstrated
nothing: the joint sampler reproduces the closed-form two-variable answer; the coupling's
source is *narrower* than its prior under joint sampling (the opposite of 7a's propagate
fingerprint); and — with PyMC — a fitted surrogate pulls both ends of the model together and
the chain actually mixed.


In [ ]:
import numpy as _np

# 1. The joint sampler reproduced the closed form (Step 2).
_err_mean = abs(float(_np.mean(samples["x"])) - float(exact_mean[0]))
_err_sd = abs(float(_np.std(samples["x"], ddof=1)) - float(exact_sd[0]))
assert _err_mean < 0.08, f"joint mean off the closed form by {_err_mean:.4f}"
assert _err_sd < 0.12 * float(exact_sd[0]), f"joint sd off the closed form by {_err_sd:.4f}"

# 2. Under JOINT sampling the coupling's SOURCE is narrower than its prior. Under
#    propagate (7a) it is untouched. This single number distinguishes the two methods,
#    and would fail if `joint` ever silently degraded into propagation.
_sd_x = float(_np.std(samples["x"], ddof=1))
# Strictly narrower, not "much" narrower: at this spec's sigma the closed form says the
# source goes from 0.500 to 0.469, a 6% effect. Asserting a big drop would be asserting a
# property of these particular numbers rather than of the method. What distinguishes the
# two methods is the DIRECTION plus the match to the closed form checked just above —
# under propagate (7a) std(x) sits at its prior exactly, forever.
assert _sd_x < sx, (
    f"std(x) = {_sd_x:.3f} is not below its prior {sx} — under joint sampling the coupling "
    "must inform its source, however slightly; under propagate it would be untouched"
)
assert _sd_x < float(exact_sd[0]) * 1.12, "std(x) drifted from the closed-form joint value"

if not PYMC_AVAILABLE:
    print(f"\n[T7b self-check OK] joint matches the closed form (mean err {_err_mean:.3f}, "
          f"sd err {_err_sd:.3f}); std(x)={_sd_x:.3f} < prior {sx} (closed form "
          f"{float(exact_sd[0]):.3f}) so the source was informed. Steps 3-6 skipped "
          f"per preflight (PyMC absent).")
else:
    # 3. NUTS agrees with the random walk — two implementations of one density (Step 3).
    assert abs(float(nuts_s["x"].mean()) - float(rw_s["x"].mean())) < 0.08, (
        "NUTS and the random walk no longer encode the same density"
    )
    # 4. The fitted surrogate drove the inference: both ends moved (Step 5).
    _af = _np.asarray(samples_f["a"], dtype=float).ravel()
    _bf = _np.asarray(samples_f["b"], dtype=float).ravel()
    _yf = _np.asarray(samples_f["y"], dtype=float).ravel()
    assert not diag_f["poorly_mixed"], (
        f"the fitted-surrogate chain did not mix: {diag_f['poorly_mixed']} — see Step 6"
    )
    assert _af.mean() > 1.15 and _bf.mean() > 1.15, (
        f"a={_af.mean():.3f}, b={_bf.mean():.3f} barely moved off their prior mean of 1.0"
    )
    assert _yf.mean() < 3.95, f"y={_yf.mean():.3f} did not come down from its prior mean of 4.0"
    assert abs(_af.mean() + _bf.mean() - _yf.mean()) < 0.5, (
        f"a+b={_af.mean() + _bf.mean():.3f} and y={_yf.mean():.3f} did not reconcile"
    )
    print(f"\n[T7b self-check OK] joint matches the closed form; std(x)={_sd_x:.3f} < prior "
          f"{sx} (source informed, unlike propagate); NUTS==random walk; fitted surrogate "
          f"met at a+b={_af.mean() + _bf.mean():.3f} vs y={_yf.mean():.3f}, worst ESS "
          f"{min(diag_f['ess'].values()):.0f}.")
